In [1]:
%pip install -U langgraph

Note: you may need to restart the kernel to use updated packages.


In [3]:
import langgraph
from importlib.metadata import version

print("LangGraph imported successfully!")
print("LangGraph version:", version("langgraph"))

LangGraph imported successfully!
LangGraph version: 1.2.11


In [229]:
from typing import TypedDict, Optional, List, Dict, Any

class CognitiveState(TypedDict, total=False):

    # Patient
    patient_id: str
    current_time: str

    # Patient repository
    patient: Dict[str, Any]
    memories: List[Dict[str, Any]]
    routines: List[Dict[str, Any]]
    goals: List[Dict[str, Any]]

    # Agent decisions
    selected_goal: Dict[str, Any]
    current_routine: Dict[str, Any]

    # Game
    mission: Dict[str, Any]

    # Content
    content: Dict[str, Any]

    # Patient interaction
    simulated_response: Dict[str, Any]
    patient_response: Dict[str, Any]

    # Adaptation
    current_difficulty: str
    next_difficulty: str

    # Final activity
    next_activity: Dict[str, Any]

    # Activity result
    activity_result: Dict[str, Any]

In [230]:
state: CognitiveState = {
    "patient_id": "lakshmi_001",
    "current_time": "10:30",
    "current_difficulty": "very_easy"
}

print(state)
print("CognitiveState created successfully!")

{'patient_id': 'lakshmi_001', 'current_time': '10:30', 'current_difficulty': 'very_easy'}
CognitiveState created successfully!


In [231]:
from datetime import datetime


def goal_node(state: CognitiveState):

    print("→ Goal Agent")

    # For now, use the existing GoalAgent logic
    goals = goal_agent.get_active_goals(
        state["patient_id"]
    )

    if not goals:
        return {
            "goals": [],
            "selected_goal": {}
        }

    selected_goal = goal_agent.select_goal(
        patient_id=state["patient_id"],
        current_time=state["current_time"]
    )

    return {
        "goals": [
            {
                "goal_id": g.goal_id,
                "patient_id": g.patient_id,
                "goal_type": g.goal_type,
                "priority": g.priority,
                "description": g.description,
                "source": g.source,
                "active": g.active
            }
            for g in goals
        ],
        "selected_goal": {
            "goal_id": selected_goal.goal_id,
            "patient_id": selected_goal.patient_id,
            "goal_type": selected_goal.goal_type,
            "priority": selected_goal.priority,
            "description": selected_goal.description,
            "source": selected_goal.source,
            "active": selected_goal.active
        }
    }

In [232]:
def routine_node(state: CognitiveState):

    print("→ Routine Agent")

    routines_data = repository.get_patient_routines(
        state["patient_id"]
    )

    current_routine = None
    closest_difference = None

    current_time = datetime.strptime(
        state["current_time"],
        "%H:%M"
    ).time()

    for routine in routines_data:

        if not routine.get("enabled", True):
            continue

        scheduled_time = routine["scheduled_time"]

        # Firestore may return a string
        if isinstance(scheduled_time, str):
            scheduled_time = datetime.strptime(
                scheduled_time,
                "%H:%M"
            ).time()

        difference = abs(
            (
                datetime.combine(
                    datetime.today(),
                    scheduled_time
                )
                -
                datetime.combine(
                    datetime.today(),
                    current_time
                )
            ).total_seconds()
        ) / 60

        if difference <= 20:

            if (
                closest_difference is None
                or difference < closest_difference
            ):
                current_routine = routine
                closest_difference = difference

    return {
        "routines": routines_data,
        "current_routine": current_routine or {}
    }

In [233]:
def game_node(state: CognitiveState):

    print("→ Game Agent")

    routine = state.get("current_routine")

    if not routine:
        return {
            "mission": {}
        }

    mission = game_agent.create_routine_mission(
        routine,
        difficulty=state.get(
            "current_difficulty",
            "very_easy"
        )
    )

    return {
        "mission": mission
    }

In [234]:
def content_node(state: CognitiveState):

    print("→ Content Agent")

    mission = state.get("mission")

    if not mission:
        return {
            "content": {}
        }

    npc_content = content_agent.generate_npc_dialogue(
        mission
    )

    activity_content = content_agent.generate_activity_text(
        mission
    )

    return {
        "content": {
            "npc": npc_content,
            "activity": activity_content
        }
    }

In [235]:
from langgraph.graph import StateGraph, START, END

builder = StateGraph(CognitiveState)

builder.add_node("goal", goal_node)
builder.add_node("routine", routine_node)
builder.add_node("game", game_node)
builder.add_node("content", content_node)

builder.add_edge(START, "goal")
builder.add_edge("goal", "routine")
builder.add_edge("routine", "game")
builder.add_edge("game", "content")
builder.add_edge("content", END)

cognitive_graph = builder.compile()

print("Cognitive graph compiled successfully!")

Cognitive graph compiled successfully!


In [236]:
initial_state = {
    "patient_id": "lakshmi_001",
    "current_time": "10:30",
    "current_difficulty": "very_easy",

    "simulated_response": {
        "answer": "Watch television",
        "response_time": 40,
        "hints_used": 2,
        "independent": False,
        "action_completed": False
    }
}

In [237]:
print(initial_state)

{'patient_id': 'lakshmi_001', 'current_time': '10:30', 'current_difficulty': 'very_easy', 'simulated_response': {'answer': 'Watch television', 'response_time': 40, 'hints_used': 2, 'independent': False, 'action_completed': False}}


In [238]:
import firebase_admin
from firebase_admin import credentials, firestore

SERVICE_ACCOUNT_PATH = "firebase/serviceAccountKey.json.json"

if not firebase_admin._apps:
    cred = credentials.Certificate(SERVICE_ACCOUNT_PATH)
    firebase_admin.initialize_app(cred)

db = firestore.client()

print("Firebase connected successfully!")

Firebase connected successfully!


In [239]:
class PatientRepository:

    def __init__(self, db):
        self.db = db

    def get_patient(self, patient_id):
        doc = self.db.collection("patients").document(patient_id).get()

        if not doc.exists:
            return None

        return doc.to_dict()

    def get_patient_memories(self, patient_id):
        docs = (
            self.db.collection("patient_memories")
            .where("patient_id", "==", patient_id)
            .stream()
        )

        return [doc.to_dict() for doc in docs]

    def get_patient_routines(self, patient_id):
        docs = (
            self.db.collection("routines")
            .where("patient_id", "==", patient_id)
            .stream()
        )

        return [doc.to_dict() for doc in docs]

    def get_patient_goals(self, patient_id):
        docs = (
            self.db.collection("goals")
            .where("patient_id", "==", patient_id)
            .stream()
        )

        return [doc.to_dict() for doc in docs]


repository = PatientRepository(db)

print("Patient Repository ready!")

Patient Repository ready!


In [240]:
from dataclasses import dataclass


@dataclass
class CognitiveGoal:

    goal_id: str
    patient_id: str
    goal_type: str
    priority: int
    description: str
    source: str
    active: bool = True


class GoalAgent:

    def __init__(self, repository):
        self.repository = repository

    def get_active_goals(self, patient_id):

        goals_data = self.repository.get_patient_goals(
            patient_id
        )

        goals = []

        for data in goals_data:

            if not data.get("active", True):
                continue

            goals.append(
                CognitiveGoal(
                    goal_id=data["goal_id"],
                    patient_id=data["patient_id"],
                    goal_type=data["goal_type"],
                    priority=data["priority"],
                    description=data["description"],
                    source=data["source"],
                    active=data.get("active", True)
                )
            )

        return goals

    def select_goal(
        self,
        patient_id,
        current_time,
        current_routines=None
    ):

        goals = self.get_active_goals(patient_id)

        if not goals:
            return None

        if current_routines:

            routine_type = current_routines.get(
                "routine_type"
            )

            matching_goals = [
                goal
                for goal in goals
                if (
                    routine_type == "hydration"
                    and goal.goal_type == "routine_recall"
                )
            ]

            if matching_goals:

                matching_goals.sort(
                    key=lambda goal: goal.priority,
                    reverse=True
                )

                return matching_goals[0]

        goals.sort(
            key=lambda goal: goal.priority,
            reverse=True
        )

        return goals[0]

In [241]:
goal_agent = GoalAgent(repository)

print("Goal Agent ready!")

Goal Agent ready!


In [242]:
class GameAgent:

    def create_routine_mission(
        self,
        routine,
        difficulty="very_easy"
    ):

        if routine["routine_type"] != "hydration":
            raise ValueError(
                "Interactive missions currently support hydration routines."
            )

        return {
            "mission_id": f"mission_{routine['routine_id']}",
            "mission_type": "routine_recall",
            "difficulty": difficulty,

            "routine": {
                "routine_id": routine["routine_id"],
                "title": routine["title"],
                "scheduled_time": routine["scheduled_time"]
            },

            "world": {
                "location": "home",
                "starting_location": "living_room",
                "target_location": "kitchen"
            },

            "npc": {
                "name": "Anu",
                "role": "family_member",
                "opening_dialogue":
                    "Amma, I think we have something to do."
            },

            "objective": {
                "description":
                    "Remember what you usually do around this time.",
                "target_object": "water_bottle",
                "target_action": "drink_water"
            },

            "steps": [
                "talk_to_npc",
                "remember_routine",
                "walk_to_kitchen",
                "find_water_bottle",
                "identify_object",
                "perform_action"
            ],

            "cognitive_targets": [
                "routine_recall",
                "time_orientation",
                "attention",
                "object_recognition",
                "action_recall"
            ],

            "completion": {
                "real_world_action_required": True,
                "cognitive_response_required": True
            }
        }


game_agent = GameAgent()

print("Game Agent ready!")

Game Agent ready!


In [243]:
class ContentAgent:

    def generate_npc_dialogue(self, mission):

        npc_name = mission["npc"]["name"]
        target_object = mission["objective"]["target_object"]

        return {
            "npc_name": npc_name,
            "opening": mission["npc"]["opening_dialogue"],
            "hint": (
                "Can you remember what we usually do "
                "around this time?"
            ),
            "object_prompt": (
                f"Can you find the "
                f"{target_object.replace('_', ' ')}?"
            ),
            "success": (
                "Well done! You remembered what to do."
            )
        }

    def generate_activity_text(self, mission):

        return {
            "title": "Daily Routine Mission",

            "instruction":
                mission["objective"]["description"],

            "goal": (
                f"Find the "
                f"{mission['objective']['target_object'].replace('_', ' ')} "
                "and complete the activity."
            ),

            "steps": mission["steps"]
        }


content_agent = ContentAgent()

print("Content Agent ready!")

Content Agent ready!


In [244]:
result = cognitive_graph.invoke(initial_state)

print("\nGraph execution completed!")

→ Goal Agent
→ Routine Agent
→ Game Agent
→ Content Agent

Graph execution completed!


In [245]:
print("\nSELECTED GOAL:")
print(result["selected_goal"])

print("\nCURRENT ROUTINE:")
print(result["current_routine"])

print("\nMISSION:")
print(result["mission"])

print("\nCONTENT:")
print(result["content"])



SELECTED GOAL:
{'goal_id': 'goal_001', 'patient_id': 'lakshmi_001', 'goal_type': 'routine_recall', 'priority': 10, 'description': 'Help the patient remember daily routines.', 'source': 'caregiver', 'active': True}

CURRENT ROUTINE:
{'title': 'Drink water', 'scheduled_time': '10:30', 'routine_type': 'hydration', 'description': 'Drink water during the morning.', 'enabled': True, 'routine_id': 'water_1030', 'patient_id': 'lakshmi_001'}

MISSION:
{'mission_id': 'mission_water_1030', 'mission_type': 'routine_recall', 'difficulty': 'very_easy', 'routine': {'routine_id': 'water_1030', 'title': 'Drink water', 'scheduled_time': '10:30'}, 'world': {'location': 'home', 'starting_location': 'living_room', 'target_location': 'kitchen'}, 'npc': {'name': 'Anu', 'role': 'family_member', 'opening_dialogue': 'Amma, I think we have something to do.'}, 'objective': {'description': 'Remember what you usually do around this time.', 'target_object': 'water_bottle', 'target_action': 'drink_water'}, 'steps': 

In [246]:
result = cognitive_graph.invoke(initial_state)

print("\nGraph execution completed!")
print("\nFinal state:")
print(result)

→ Goal Agent
→ Routine Agent
→ Game Agent
→ Content Agent

Graph execution completed!

Final state:
{'patient_id': 'lakshmi_001', 'current_time': '10:30', 'routines': [{'title': 'Drink water', 'scheduled_time': '08:30', 'routine_type': 'hydration', 'description': 'Drink water after breakfast.', 'enabled': True, 'routine_id': 'water_0830', 'patient_id': 'lakshmi_001'}, {'title': 'Drink water', 'scheduled_time': '10:30', 'routine_type': 'hydration', 'description': 'Drink water during the morning.', 'enabled': True, 'routine_id': 'water_1030', 'patient_id': 'lakshmi_001'}, {'title': 'Drink water', 'scheduled_time': '13:00', 'routine_type': 'hydration', 'description': 'Drink water around lunchtime.', 'enabled': True, 'routine_id': 'water_1300', 'patient_id': 'lakshmi_001'}, {'title': 'Drink water', 'scheduled_time': '15:30', 'routine_type': 'hydration', 'description': 'Drink water during the afternoon.', 'enabled': True, 'routine_id': 'water_1530', 'patient_id': 'lakshmi_001'}, {'title': '

In [247]:
def patient_interaction_node(state: CognitiveState):
    print("→ Patient Interaction")

    # Use a simulated response supplied in the initial state.
    # If none is supplied, use the default successful response.
    response = state.get(
        "simulated_response",
        {
            "answer": "Drink water",
            "response_time": 6,
            "hints_used": 0,
            "independent": True,
            "action_completed": True
        }
    )

    print("Patient response:", response)

    return {
        "patient_response": response
    }

print("Patient Interaction node updated!")

Patient Interaction node updated!


In [248]:
def performance_node(state: CognitiveState):

    print("→ Performance Evaluation")

    response = state["patient_response"]
    mission = state["mission"]

    correct = (
        response["answer"].strip().lower()
        == "drink water"
    )

    result = {
        "patient_id": state["patient_id"],
        "activity_id": mission["mission_id"],
        "routine_id": mission["routine"]["routine_id"],
        "difficulty": mission["difficulty"],
        "correct": correct,
        "response_time": response["response_time"],
        "hints_used": response["hints_used"],
        "independent": response["independent"],
        "action_completed": response["action_completed"]
    }

    return {
        "activity_result": result
    }

print("Performance Evaluation node ready!")

Performance Evaluation node ready!


In [249]:
class AdaptationAgent:

    levels = [
        "very_easy",
        "easy",
        "medium",
        "hard"
    ]

    def update(
        self,
        difficulty,
        correct,
        hints_used,
        response_time,
        independent
    ):

        if difficulty not in self.levels:
            raise ValueError(
                f"Unknown difficulty level: {difficulty}"
            )

        index = self.levels.index(difficulty)

        if (
            correct
            and independent
            and hints_used == 0
            and response_time < 10
        ):
            index += 1

        elif (
            not correct
            or hints_used >= 2
            or response_time > 30
        ):
            index -= 1

        index = max(
            0,
            min(index, len(self.levels) - 1)
        )

        return self.levels[index]

In [250]:
adaptation_agent = AdaptationAgent()

print("Adaptation Agent ready!")

Adaptation Agent ready!


In [251]:
def adaptation_node(state: CognitiveState):

    print("→ Adaptation Agent")

    result = state["activity_result"]

    next_difficulty = adaptation_agent.update(
        difficulty=result["difficulty"],
        correct=result["correct"],
        hints_used=result["hints_used"],
        response_time=result["response_time"],
        independent=result["independent"]
    )

    return {
        "next_difficulty": next_difficulty
    }

In [252]:
from datetime import datetime, timezone

def save_result_node(state: CognitiveState):
    print("→ Saving Activity Result")

    result = state["activity_result"]

    result_id = (
        f"{result['activity_id']}_"
        f"{datetime.now(timezone.utc).strftime('%Y%m%d%H%M%S')}"
    )

    repository.save_activity_result(
        result_id,
        result
    )

    print(f"Activity result saved: {result_id}")

    return {
        "activity_result": {
            **result,
            "result_id": result_id
        }
    }

In [253]:
def adaptation_node(state: CognitiveState):

    print("→ Adaptation Agent")

    result = state["activity_result"]

    next_difficulty = adaptation_agent.update(
        difficulty=result["difficulty"],
        correct=result["correct"],
        hints_used=result["hints_used"],
        response_time=result["response_time"],
        independent=result["independent"]
    )

    print(f"Next difficulty: {next_difficulty}")

    return {
        "next_difficulty": next_difficulty
    }


print("Adaptation node ready!")

Adaptation node ready!


In [254]:
builder.add_node(
    "patient_interaction",
    patient_interaction_node
)

builder.add_node(
    "performance",
    performance_node
)

builder.add_node(
    "save_result",
    save_result_node
)

builder.add_node(
    "adaptation",
    adaptation_node
)

print("New nodes added!")

Adding a node to a graph that has already been compiled. This will not be reflected in the compiled graph.
Adding a node to a graph that has already been compiled. This will not be reflected in the compiled graph.
Adding a node to a graph that has already been compiled. This will not be reflected in the compiled graph.
Adding a node to a graph that has already been compiled. This will not be reflected in the compiled graph.


New nodes added!


In [255]:
builder.add_edge(
    "content",
    "patient_interaction"
)

builder.add_edge(
    "patient_interaction",
    "performance"
)

builder.add_edge(
    "performance",
    "save_result"
)

builder.add_edge(
    "save_result",
    "adaptation"
)

builder.add_edge(
    "adaptation",
    END
)

print("New edges added!")

Adding an edge to a graph that has already been compiled. This will not be reflected in the compiled graph.
Adding an edge to a graph that has already been compiled. This will not be reflected in the compiled graph.
Adding an edge to a graph that has already been compiled. This will not be reflected in the compiled graph.
Adding an edge to a graph that has already been compiled. This will not be reflected in the compiled graph.
Adding an edge to a graph that has already been compiled. This will not be reflected in the compiled graph.


New edges added!


In [256]:
cognitive_graph = builder.compile()

print("Updated cognitive graph compiled!")

Updated cognitive graph compiled!


In [257]:
from datetime import datetime, timezone

def save_activity_result(self, result_id: str, data: dict):
    data = data.copy()

    data["created_at"] = datetime.now(timezone.utc).isoformat()

    self.db.collection("activity_results").document(
        result_id
    ).set(
        data,
        merge=True
    )

PatientRepository.save_activity_result = save_activity_result

print("PatientRepository activity-result saving enabled!")

PatientRepository activity-result saving enabled!


In [258]:
print(hasattr(repository, "save_activity_result"))

True


In [259]:
test_result = {
    "patient_id": "lakshmi_001",
    "activity_id": "test_activity",
    "correct": True
}

repository.save_activity_result(
    "test_result_001",
    test_result
)

print("Test activity result saved successfully!")

Test activity result saved successfully!


In [260]:
result = cognitive_graph.invoke(initial_state)

print("\n==============================")
print("ADAPTIVE COGNITIVE LOOP DONE")
print("==============================")

print("\nNEXT DIFFICULTY:")
print(result["next_difficulty"])

print("\nACTIVITY RESULT:")
print(result["activity_result"])

→ Goal Agent
→ Routine Agent
→ Game Agent
→ Content Agent
→ Patient Interaction
Patient response: {'answer': 'Watch television', 'response_time': 40, 'hints_used': 2, 'independent': False, 'action_completed': False}
→ Performance Evaluation
→ Saving Activity Result
Activity result saved: mission_water_1030_20260911044921
→ Adaptation Agent
Next difficulty: very_easy

ADAPTIVE COGNITIVE LOOP DONE

NEXT DIFFICULTY:
very_easy

ACTIVITY RESULT:
{'patient_id': 'lakshmi_001', 'activity_id': 'mission_water_1030', 'routine_id': 'water_1030', 'difficulty': 'very_easy', 'correct': False, 'response_time': 40, 'hints_used': 2, 'independent': False, 'action_completed': False, 'result_id': 'mission_water_1030_20260911044921'}


In [261]:
result = cognitive_graph.invoke(initial_state)

→ Goal Agent
→ Routine Agent
→ Game Agent
→ Content Agent
→ Patient Interaction
Patient response: {'answer': 'Watch television', 'response_time': 40, 'hints_used': 2, 'independent': False, 'action_completed': False}
→ Performance Evaluation
→ Saving Activity Result
Activity result saved: mission_water_1030_20260911044922
→ Adaptation Agent
Next difficulty: very_easy


In [262]:
result = cognitive_graph.invoke(initial_state)

print("\nNEXT DIFFICULTY:")
print(result["next_difficulty"])

→ Goal Agent
→ Routine Agent
→ Game Agent
→ Content Agent
→ Patient Interaction
Patient response: {'answer': 'Watch television', 'response_time': 40, 'hints_used': 2, 'independent': False, 'action_completed': False}
→ Performance Evaluation
→ Saving Activity Result
Activity result saved: mission_water_1030_20260911044923
→ Adaptation Agent
Next difficulty: very_easy

NEXT DIFFICULTY:
very_easy


In [263]:
print("CURRENT INITIAL STATE:")
print(initial_state)

print("\nSIMULATED RESPONSE:")
print(initial_state.get("simulated_response"))

CURRENT INITIAL STATE:
{'patient_id': 'lakshmi_001', 'current_time': '10:30', 'current_difficulty': 'very_easy', 'simulated_response': {'answer': 'Watch television', 'response_time': 40, 'hints_used': 2, 'independent': False, 'action_completed': False}}

SIMULATED RESPONSE:
{'answer': 'Watch television', 'response_time': 40, 'hints_used': 2, 'independent': False, 'action_completed': False}


In [264]:
test_state = {
    "patient_id": "lakshmi_001",
    "current_time": "10:30",
    "current_difficulty": "very_easy",
    "simulated_response": {
        "answer": "Watch television",
        "response_time": 40,
        "hints_used": 2,
        "independent": False,
        "action_completed": False
    }
}

print("TEST STATE:")
print(test_state)

result = cognitive_graph.invoke(test_state)

TEST STATE:
{'patient_id': 'lakshmi_001', 'current_time': '10:30', 'current_difficulty': 'very_easy', 'simulated_response': {'answer': 'Watch television', 'response_time': 40, 'hints_used': 2, 'independent': False, 'action_completed': False}}
→ Goal Agent
→ Routine Agent
→ Game Agent
→ Content Agent
→ Patient Interaction
Patient response: {'answer': 'Watch television', 'response_time': 40, 'hints_used': 2, 'independent': False, 'action_completed': False}
→ Performance Evaluation
→ Saving Activity Result
Activity result saved: mission_water_1030_20260911044925
→ Adaptation Agent
Next difficulty: very_easy


In [265]:
print("Current Python function:")
print(patient_interaction_node)

print("\nFunction source:")
import inspect
print(inspect.getsource(patient_interaction_node))

Current Python function:
<function patient_interaction_node at 0x000001971DB31C60>

Function source:
def patient_interaction_node(state: CognitiveState):
    print("→ Patient Interaction")

    # Use a simulated response supplied in the initial state.
    # If none is supplied, use the default successful response.
    response = state.get(
        "simulated_response",
        {
            "answer": "Drink water",
            "response_time": 6,
            "hints_used": 0,
            "independent": True,
            "action_completed": True
        }
    )

    print("Patient response:", response)

    return {
        "patient_response": response
    }



In [273]:
from langgraph.graph import StateGraph, START, END

new_builder = StateGraph(CognitiveState)

new_builder.add_node("goal", goal_node)
new_builder.add_node("routine", routine_node)
new_builder.add_node("game", game_node)
new_builder.add_node("content", content_node)
new_builder.add_node("patient_interaction", patient_interaction_node)
new_builder.add_node("performance", performance_node)
new_builder.add_node("save_result", save_result_node)
new_builder.add_node("adaptation", adaptation_node)

new_builder.add_edge(START, "goal")
new_builder.add_edge("goal", "routine")
new_builder.add_edge("routine", "game")
new_builder.add_edge("game", "content")
new_builder.add_edge("content", "patient_interaction")
new_builder.add_edge("patient_interaction", "performance")
new_builder.add_edge("performance", "save_result")
new_builder.add_edge("save_result", "adaptation")
new_builder.add_edge("adaptation", END)

cognitive_graph = new_builder.compile()

print("NEW GRAPH COMPILED SUCCESSFULLY")

NEW GRAPH COMPILED SUCCESSFULLY


In [274]:
test_state = {
    "patient_id": "lakshmi_001",
    "current_time": "10:30",
    "current_difficulty": "very_easy",
    "simulated_response": {
        "answer": "Watch television",
        "response_time": 40,
        "hints_used": 2,
        "independent": False,
        "action_completed": False
    }
}

result = cognitive_graph.invoke(test_state)

→ Goal Agent
→ Routine Agent
→ Game Agent
→ Content Agent
→ Patient Interaction
Patient response: {'answer': 'Watch television', 'response_time': 40, 'hints_used': 2, 'independent': False, 'action_completed': False}
→ Performance Evaluation
→ Saving Activity Result
Activity result saved: mission_water_1030_20260911044959
→ Adaptation Agent
Next difficulty: very_easy


In [275]:
import inspect

print("NODE SOURCE:")
print(inspect.getsource(patient_interaction_node))

print("\nTESTING NODE DIRECTLY:")
direct_test = patient_interaction_node({
    "simulated_response": {
        "answer": "Watch television",
        "response_time": 40,
        "hints_used": 2,
        "independent": False,
        "action_completed": False
    }
})

print("\nDIRECT NODE OUTPUT:")
print(direct_test)

NODE SOURCE:
def patient_interaction_node(state: CognitiveState):
    print("→ Patient Interaction")

    # Use a simulated response supplied in the initial state.
    # If none is supplied, use the default successful response.
    response = state.get(
        "simulated_response",
        {
            "answer": "Drink water",
            "response_time": 6,
            "hints_used": 0,
            "independent": True,
            "action_completed": True
        }
    )

    print("Patient response:", response)

    return {
        "patient_response": response
    }


TESTING NODE DIRECTLY:
→ Patient Interaction
Patient response: {'answer': 'Watch television', 'response_time': 40, 'hints_used': 2, 'independent': False, 'action_completed': False}

DIRECT NODE OUTPUT:
{'patient_response': {'answer': 'Watch television', 'response_time': 40, 'hints_used': 2, 'independent': False, 'action_completed': False}}


In [276]:
print("NODE STORED IN BUILDER:")
print(new_builder.nodes["patient_interaction"])

NODE STORED IN BUILDER:
StateNodeSpec(runnable=patient_interaction(tags=None, recurse=True, explode_args=False, func_accepts={}), metadata=None, input_schema=<class '__main__.CognitiveState'>, retry_policy=None, cache_policy=None, is_error_handler=False, error_handler_node=None, ends=(), defer=False, timeout=None, trace_policy=None)


In [277]:
print("\nCOMPILED GRAPH NODE:")
print(cognitive_graph.nodes["patient_interaction"])


COMPILED GRAPH NODE:


In [278]:
from langgraph.graph import StateGraph, START, END

debug_builder = StateGraph(CognitiveState)

debug_builder.add_node("patient_interaction", patient_interaction_node)

debug_builder.add_edge(START, "patient_interaction")
debug_builder.add_edge("patient_interaction", END)

debug_graph = debug_builder.compile()

debug_result = debug_graph.invoke({
    "simulated_response": {
        "answer": "Watch television",
        "response_time": 40,
        "hints_used": 2,
        "independent": False,
        "action_completed": False
    }
})

print("DEBUG GRAPH RESULT:")
print(debug_result)

→ Patient Interaction
Patient response: {'answer': 'Watch television', 'response_time': 40, 'hints_used': 2, 'independent': False, 'action_completed': False}
DEBUG GRAPH RESULT:
{'simulated_response': {'answer': 'Watch television', 'response_time': 40, 'hints_used': 2, 'independent': False, 'action_completed': False}, 'patient_response': {'answer': 'Watch television', 'response_time': 40, 'hints_used': 2, 'independent': False, 'action_completed': False}}


In [279]:
test_state = {
    "patient_id": "lakshmi_001",
    "current_time": "10:30",
    "current_difficulty": "very_easy",

    "simulated_response": {
        "answer": "Watch television",
        "response_time": 40,
        "hints_used": 2,
        "independent": False,
        "action_completed": False
    }
}

result = cognitive_graph.invoke(test_state)

print("\nFINAL RESULT:")
print(result)

print("\nNEXT DIFFICULTY:")
print(result["next_difficulty"])

→ Goal Agent
→ Routine Agent
→ Game Agent
→ Content Agent
→ Patient Interaction
Patient response: {'answer': 'Watch television', 'response_time': 40, 'hints_used': 2, 'independent': False, 'action_completed': False}
→ Performance Evaluation
→ Saving Activity Result
Activity result saved: mission_water_1030_20260911045051
→ Adaptation Agent
Next difficulty: very_easy

FINAL RESULT:
{'patient_id': 'lakshmi_001', 'current_time': '10:30', 'routines': [{'title': 'Drink water', 'scheduled_time': '08:30', 'routine_type': 'hydration', 'description': 'Drink water after breakfast.', 'enabled': True, 'routine_id': 'water_0830', 'patient_id': 'lakshmi_001'}, {'title': 'Drink water', 'scheduled_time': '10:30', 'routine_type': 'hydration', 'description': 'Drink water during the morning.', 'enabled': True, 'routine_id': 'water_1030', 'patient_id': 'lakshmi_001'}, {'title': 'Drink water', 'scheduled_time': '13:00', 'routine_type': 'hydration', 'description': 'Drink water around lunchtime.', 'enabled':